In [ ]:
import base64, glob, hashlib, io, os, shlex, subprocess, tarfile
raw = base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest() == 'a2e8a92decbd4ee1034a9c8e726df6393bb73232900940f28878c925055969ea'
CODE, OUT = '/kaggle/working/code', '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)
tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz').extractall(CODE)
print('code sha256 a2e8a92decbd4ee1034a9c8e726df6393bb73232900940f28878c925055969ea from git fc2f350+local changes', flush=True)


def find(pattern):
    hits = sorted(glob.glob('/kaggle/input/**/' + pattern, recursive=True), key=len)
    return hits[0] if hits else ''


chik = [p for p in glob.glob('/kaggle/input/**/*.mat', recursive=True) if 'chikusei' in p.lower() and 'Ground' not in p]
P = dict(chikusei=sorted(chik, key=len)[0] if chik else '', pavia=find('Pavia.mat'),
         cave=os.path.dirname(os.path.dirname(os.path.dirname(find('Train/HSI/balloons.mat') or '/x/x/x/x'))),
         harvard=os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(find('Data/Train/HSI/img*.mat') or '/x/x/x/x/x')))),
         cache_chikusei='/tmp/chikusei.npy', OUT=OUT)
P = {k: shlex.quote(v) for k, v in P.items()}
for cmd in ['python train.py --dataset pavia --mat {pavia} --scale 4 --pad reflect --eval_every 10 --log_every 5 --test_log 0 --amp 0 --bs 2 --iters 10 --model puformer --dstep kronlearn --nsp 1 --mres 1 --sa 1 --unc 1 --out {OUT}/unc', 'python unc_eval.py --dataset pavia --mat {pavia} --ckpt {OUT}/unc/best_ema.pt --model puformer --dstep kronlearn --nsp 1 --mres 1 --sa 1 --unc 1 --out {OUT}/unc-eval']:
    cmd = cmd.format(**P)
    print('RUN', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=CODE)
    print('END rc', r.returncode, flush=True)
